# Team505 — Preprocessing Pipeline

**DSAI 305 — Phase 2**  
**Dataset:** NIH ChestX-ray14  
**Objective:** Build a clean, reproducible preprocessing pipeline that:

1. Loads the official metadata (`Data_Entry_2017.csv`)
2. Resolves image file paths across `images_001` to `images_012`
3. Uses the official NIH split files (`train_val_list.txt`, `test_list.txt`)
4. Creates a binary pneumonia target
5. Performs a patient-wise validation split from the train_val portion
6. Saves clean split CSVs to `data/splits/`

**Note:** The official NIH test split is kept unchanged.

## 0 — Imports & Configuration

In [1]:
import sys
print(sys.executable)

c:\Users\s-amm\Downloads\ChestX-ray14\Team505_phase2\.venv\Scripts\python.exe


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
import warnings

warnings.filterwarnings('ignore')

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Paths (relative to notebooks/ directory)
PROJECT_ROOT = Path('..').resolve()
DATA_RAW     = PROJECT_ROOT / 'data' / 'raw' / 'nih_chest_xray14'
DATA_SPLITS  = PROJECT_ROOT / 'data' / 'splits'

# Ensure output directory exists
DATA_SPLITS.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Raw data     : {DATA_RAW}')
print(f'Splits output: {DATA_SPLITS}')

Project root : C:\Users\s-amm\Downloads\ChestX-ray14\Team505_phase2
Raw data     : C:\Users\s-amm\Downloads\ChestX-ray14\Team505_phase2\data\raw\nih_chest_xray14
Splits output: C:\Users\s-amm\Downloads\ChestX-ray14\Team505_phase2\data\splits


## 1 — Load Master Metadata

In [3]:
# Load official NIH metadata
meta_path = DATA_RAW / 'Data_Entry_2017.csv'
df_raw = pd.read_csv(meta_path)

print(f'Raw metadata shape: {df_raw.shape}')
print(f'Columns: {list(df_raw.columns)}')
df_raw.head(3)

Raw metadata shape: (112120, 12)
Columns: ['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID', 'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width', 'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'Unnamed: 11']


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,0.143,NaN
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,0.143,NaN
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168,NaN


In [4]:
# Rename columns for clarity
df = df_raw[['Image Index', 'Finding Labels', 'Patient ID',
             'Patient Age', 'Patient Gender', 'View Position']].copy()

df.columns = ['image_name', 'finding_labels', 'patient_id',
              'patient_age', 'patient_gender', 'view_position']

print(f'Working dataframe shape: {df.shape}')
df.head()

Working dataframe shape: (112120, 6)


,image_name,finding_labels,patient_id,patient_age,patient_gender,view_position
0,00000001_000.png,Cardiomegaly,1,58,M,PA
1,00000001_001.png,Cardiomegaly|Emphysema,1,58,M,PA
2,00000001_002.png,Cardiomegaly|Effusion,1,58,M,PA
3,00000002_000.png,No Finding,2,81,M,PA
4,00000003_000.png,Hernia,3,81,F,PA


## 2 — Resolve Image Paths Across Subfolders

In [5]:
# Build a lookup: image_name -> full path across images_001..images_012
print('Scanning image folders...')

image_dirs = sorted(DATA_RAW.glob('images_*'))
print(f'Found {len(image_dirs)} image directories: {[d.name for d in image_dirs]}')

path_lookup = {}
for img_dir in image_dirs:
    sub_images = img_dir / 'images'  # NIH structure: images_00X/images/*.png
    search_dir = sub_images if sub_images.exists() else img_dir
    for png_path in search_dir.glob('*.png'):
        path_lookup[png_path.name] = str(png_path)

print(f'Total images found on disk: {len(path_lookup):,}')

Scanning image folders...
Found 12 image directories: ['images_001', 'images_002', 'images_003', 'images_004', 'images_005', 'images_006', 'images_007', 'images_008', 'images_009', 'images_010', 'images_011', 'images_012']
Total images found on disk: 112,120


In [6]:
# Map image paths to metadata
df['image_path'] = df['image_name'].map(path_lookup)

# Check for missing images
missing_mask = df['image_path'].isna()
n_missing = missing_mask.sum()
print(f'Images with resolved paths: {(~missing_mask).sum():,}')
print(f'Images with missing paths : {n_missing:,}')

if n_missing > 0:
    print('\nDropping rows with missing image paths...')
    df = df[~missing_mask].reset_index(drop=True)
    print(f'Dataframe shape after cleanup: {df.shape}')
else:
    print('All images resolved successfully.')

Images with resolved paths: 112,120
Images with missing paths : 0
All images resolved successfully.


## 3 — Create Binary Pneumonia Target

In [7]:
# Binary target: 1 if "Pneumonia" appears in finding labels, else 0
df['target_pneumonia'] = df['finding_labels'].str.contains(
    'Pneumonia', case=False, na=False
).astype(int)

print('=== Pneumonia Target Distribution ===')
print(df['target_pneumonia'].value_counts().to_string())
print(f'\nPositive rate: {df["target_pneumonia"].mean():.4f} '
      f'({df["target_pneumonia"].sum():,} / {len(df):,})')

=== Pneumonia Target Distribution ===
target_pneumonia
0    110689
1      1431

Positive rate: 0.0128 (1,431 / 112,120)


## 4 — Apply Official NIH Splits

In [8]:
# Load official split files
train_val_list = set(
    (DATA_RAW / 'train_val_list.txt').read_text().strip().split('\n')
)
test_list = set(
    (DATA_RAW / 'test_list.txt').read_text().strip().split('\n')
)

print(f'Official train_val images: {len(train_val_list):,}')
print(f'Official test images     : {len(test_list):,}')
print(f'Overlap                  : {len(train_val_list & test_list)}')

Official train_val images: 86,524
Official test images     : 25,596
Overlap                  : 0


In [9]:
# Assign split_source based on official lists
df['split_source'] = 'unknown'
df.loc[df['image_name'].isin(train_val_list), 'split_source'] = 'train_val'
df.loc[df['image_name'].isin(test_list), 'split_source'] = 'test'

print('=== Split Source Counts ===')
print(df['split_source'].value_counts().to_string())

# Verify no unknowns
n_unknown = (df['split_source'] == 'unknown').sum()
if n_unknown > 0:
    print(f'\n⚠ WARNING: {n_unknown} images not in either split list!')
else:
    print('\n✓ All images accounted for in official splits.')

=== Split Source Counts ===
split_source
train_val    86524
test         25596

✓ All images accounted for in official splits.


In [10]:
# Verify patient-level separation between train_val and test
train_val_patients = set(df[df['split_source'] == 'train_val']['patient_id'].unique())
test_patients = set(df[df['split_source'] == 'test']['patient_id'].unique())
patient_overlap = train_val_patients & test_patients

print(f'Train_val patients: {len(train_val_patients):,}')
print(f'Test patients     : {len(test_patients):,}')
print(f'Patient overlap   : {len(patient_overlap)}')

if len(patient_overlap) == 0:
    print('\n✓ Official split is patient-wise — no data leakage.')
else:
    print(f'\n⚠ WARNING: {len(patient_overlap)} patients appear in both splits!')

Train_val patients: 28,008
Test patients     : 2,797
Patient overlap   : 0

✓ Official split is patient-wise — no data leakage.


## 5 — Patient-Wise Train / Validation Split

From the official `train_val` portion, we create a **validation set** using patient-level grouping  
to prevent data leakage. We use an **85/15 train/val** ratio.

In [11]:
# Separate official splits
df_train_val = df[df['split_source'] == 'train_val'].copy()
df_test = df[df['split_source'] == 'test'].copy()

print(f'Train+Val pool: {len(df_train_val):,} images from {df_train_val["patient_id"].nunique():,} patients')
print(f'Test (locked) : {len(df_test):,} images from {df_test["patient_id"].nunique():,} patients')

Train+Val pool: 86,524 images from 28,008 patients
Test (locked) : 25,596 images from 2,797 patients


In [12]:
# Patient-wise split: 85% train, 15% validation
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=RANDOM_SEED)

train_idx, val_idx = next(
    gss.split(df_train_val, groups=df_train_val['patient_id'])
)

df_train = df_train_val.iloc[train_idx].copy()
df_val   = df_train_val.iloc[val_idx].copy()

# Update split_source
df_train['split_source'] = 'train'
df_val['split_source']   = 'val'
df_test['split_source']  = 'test'

print(f'Train: {len(df_train):,} images, {df_train["patient_id"].nunique():,} patients')
print(f'Val  : {len(df_val):,} images, {df_val["patient_id"].nunique():,} patients')
print(f'Test : {len(df_test):,} images, {df_test["patient_id"].nunique():,} patients')

Train: 73,916 images, 23,806 patients
Val  : 12,608 images, 4,202 patients
Test : 25,596 images, 2,797 patients


In [13]:
# Verify zero patient overlap across all three splits
train_pids = set(df_train['patient_id'].unique())
val_pids   = set(df_val['patient_id'].unique())
test_pids  = set(df_test['patient_id'].unique())

print('Patient overlap checks:')
print(f'  Train ∩ Val  : {len(train_pids & val_pids)}')
print(f'  Train ∩ Test : {len(train_pids & test_pids)}')
print(f'  Val   ∩ Test : {len(val_pids & test_pids)}')

assert len(train_pids & val_pids) == 0,  'Train-Val patient leak!'
assert len(train_pids & test_pids) == 0, 'Train-Test patient leak!'
assert len(val_pids & test_pids) == 0,   'Val-Test patient leak!'
print('\n✓ All splits are patient-disjoint. No data leakage.')

Patient overlap checks:
  Train ∩ Val  : 0
  Train ∩ Test : 0
  Val   ∩ Test : 0

✓ All splits are patient-disjoint. No data leakage.


## 6 — Save Split CSVs

In [14]:
# Define output columns
output_cols = ['image_name', 'image_path', 'patient_id',
               'finding_labels', 'target_pneumonia', 'split_source',
               'patient_age', 'patient_gender', 'view_position']

# Save individual split files
df_train[output_cols].to_csv(DATA_SPLITS / 'train.csv', index=False)
df_val[output_cols].to_csv(DATA_SPLITS / 'val.csv', index=False)
df_test[output_cols].to_csv(DATA_SPLITS / 'test.csv', index=False)

# Save train_dev = train + val (the full official train_val with our labels)
df_train_dev = pd.concat([df_train, df_val], ignore_index=True)
df_train_dev['split_source'] = 'train_dev'
df_train_dev[output_cols].to_csv(DATA_SPLITS / 'train_dev.csv', index=False)

print('Saved split files:')
for csv_file in sorted(DATA_SPLITS.glob('*.csv')):
    n_rows = len(pd.read_csv(csv_file))
    print(f'  {csv_file.name:20s} → {n_rows:,} rows')

Saved split files:
  test.csv             → 25,596 rows
  train.csv            → 73,916 rows
  train_dev.csv        → 86,524 rows
  val.csv              → 12,608 rows


## 7 — Final Summary Statistics

In [15]:
print('=' * 60)
print('PREPROCESSING SUMMARY')
print('=' * 60)

total = len(df_train) + len(df_val) + len(df_test)
total_pos = df['target_pneumonia'].sum()
total_neg = len(df) - total_pos

print(f'\nTotal usable images    : {total:,}')
print(f'Pneumonia positives    : {total_pos:,}')
print(f'Non-pneumonia negatives: {total_neg:,}')
print(f'Positive rate          : {total_pos/total:.4f}')

print(f'\n{"Split":<12} {"Images":>10} {"Patients":>10} {"Pos":>8} {"Neg":>8} {"Pos%":>8}')
print('-' * 58)

for name, split_df in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    n = len(split_df)
    p = split_df['patient_id'].nunique()
    pos = split_df['target_pneumonia'].sum()
    neg = n - pos
    pct = pos / n * 100 if n > 0 else 0
    print(f'{name:<12} {n:>10,} {p:>10,} {pos:>8,} {neg:>8,} {pct:>7.2f}%')

print('-' * 58)
print(f'{"TOTAL":<12} {total:>10,} {"":>10} {total_pos:>8,} {total_neg:>8,}')

print(f'\n✓ Splits saved to: {DATA_SPLITS}')
print('✓ Preprocessing complete. Ready for EDA and model training.')

PREPROCESSING SUMMARY

Total usable images    : 112,120
Pneumonia positives    : 1,431
Non-pneumonia negatives: 110,689
Positive rate          : 0.0128

Split            Images   Patients      Pos      Neg     Pos%
----------------------------------------------------------
Train            73,916     23,806      742   73,174    1.00%
Val              12,608      4,202      134   12,474    1.06%
Test             25,596      2,797      555   25,041    2.17%
----------------------------------------------------------
TOTAL           112,120               1,431  110,689

✓ Splits saved to: C:\Users\s-amm\Downloads\ChestX-ray14\Team505_phase2\data\splits
✓ Preprocessing complete. Ready for EDA and model training.
